In [ ]:
#| default_exp lock

In [ ]:
#| hide
from fastcore.test import *

A thin app gets its Python packages from a lock: a pinned project (`pyproject.toml`, `uv.lock`, `.python-version`) and a signed `manifest.json`. A new lock reaches people with no new app build. `contract/api-1.md` gives the formats; this module makes and checks locks, smoke-tests one as the launcher builds it, signs it with SSH keys, and publishes it on the app's GitHub releases.

Everything app-specific comes from a `Thin` spec. An app adds its own checks in a hooks file (`Thin.hooks`): `extra_reqs()` gives each extra's requirements, `checks(s, base, port)` runs against the first server, and `smoke(s)` runs more checks after it.

In [ ]:
#| export
from __future__ import annotations
import argparse, hashlib, importlib.util, json, os, re, shutil, socket, subprocess, sys, tempfile, threading, time, tomllib, urllib.error, urllib.parse, urllib.request, uuid
from datetime import datetime, timedelta, timezone
from types import SimpleNamespace
from fastcore.all import Path
from kavacha.thin import Thin
from kavacha.host import host_env

### Small helpers

In [ ]:
#| export
FILES = ('pyproject.toml', 'uv.lock', '.python-version')
PYPI = 'https://pypi.org/simple'

def sh(*cmd, check=True, **kw):
    "Run `cmd`; its stdout, stripped."
    p = subprocess.run([str(c) for c in cmd], text=True, capture_output=True, **kw)
    if check and p.returncode: raise SystemExit(f"{Path(str(cmd[0])).name} {' '.join(map(str, cmd[1:3]))} failed: {(p.stderr or p.stdout).strip()[-2000:]}")
    return p.stdout.strip()

def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def now(): return datetime.now(timezone.utc)
def iso(t): return t.strftime('%Y-%m-%dT%H:%M:%SZ')
def dumps(o): return json.dumps(o, indent=1, sort_keys=True) + '\n'
def uv_version(uv): return sh(uv, '--version').split()[1]
def asset_name(f): return 'python-version' if f == '.python-version' else f

### The lock hash

The lock hash is sha256 over the name and the bytes of each lock file, then `uv` and the uv version. The launcher computes the same value; `contract/fixtures/lock_hash` pins one vector for every side.

In [ ]:
#| export
def lock_hash(d, uv):
    "sha256 over the names and contents of the lock files, then the uv version."
    h = hashlib.sha256()
    for f in FILES: h.update(f.encode()); h.update((Path(d)/f).read_bytes())
    h.update(b'uv'); h.update(uv.encode())
    return h.hexdigest()

In [ ]:
import tempfile
d = Path(tempfile.mkdtemp())
(d/'pyproject.toml').write_text('p'); (d/'uv.lock').write_text('l'); (d/'.python-version').write_text('3.12.16\n')
test_eq(lock_hash(d, '0.13.6'), '2b6ad4d75869cd770d5cf35be37c47dc40fe7cfce61490729959b38333e9472a')

### Platforms and wheels

A platform such as `darwin-arm64` gives the uv environment marker and the wheel tags that install there.

In [ ]:
#| export
_PLAT = {'darwin-arm64': ("sys_platform == 'darwin' and platform_machine == 'arm64'", ('arm64', 'universal2'), 'macosx'),
         'linux-x64': ("sys_platform == 'linux' and platform_machine == 'x86_64'", ('x86_64',), 'manylinux'),
         'linux-arm64': ("sys_platform == 'linux' and platform_machine == 'aarch64'", ('aarch64',), 'manylinux'),
         'win32-x64': ("sys_platform == 'win32' and platform_machine == 'AMD64'", ('amd64',), 'win')}

def marker(platform): return _PLAT[platform][0]

def wheel_fits(fn, python='3.13.18', platform='darwin-arm64'):
    "Whether a wheel filename installs on `python` and `platform`."
    if fn.count('-') < 4: return False
    cp = 'cp' + ''.join(python.split('.')[:2])
    py, abi, plat = fn[:-4].split('-')[-3:]
    abis, (_, arch, os_) = set(abi.split('.')), _PLAT[platform]
    pyok = lambda p: p in (cp, 'py3', 'py' + cp[2:]) or ('abi3' in abis and re.fullmatch(r'cp3\d+', p) and int(p[3:]) <= int(cp[3:]))
    return (any(map(pyok, py.split('.'))) and bool(abis & {cp, 'abi3', 'none'})
            and any(p == 'any' or (p.startswith(os_) and p.endswith(arch)) for p in plat.split('.')))

In [ ]:
test_eq([wheel_fits(f) for f in ('a-1.0-cp313-cp313-macosx_11_0_arm64.whl', 'a-1.0-cp39-abi3-macosx_10_12_universal2.whl', 'a-1.0-cp312-cp312-macosx_11_0_arm64.whl')], [True, True, False])
test_eq(wheel_fits('a-1.0-cp313-cp313-win_amd64.whl', platform='win32-x64'), True)
test_eq(wheel_fits('a-1.0-cp313-cp313-manylinux_2_17_x86_64.whl', platform='linux-x64'), True)

### The lock tool

`Lock` holds one app's spec and the folders that follow from it. Its methods are the lock tool's commands. `Lock.hooks` loads the app's hooks file once.

In [ ]:
#| export
class Lock:
    "The lock tool for one thin app."
    def __init__(self, t: Thin, root=None):
        self.t, self.root = t, Path(root or getattr(t, 'root', '.')).resolve()
        self.lock, self.keys_pub = self.root/t.lock_dir, self.root/t.keys
        self.repo, self.dl = t.repo, f'https://github.com/{t.repo}/releases/download'
        self.minor, self.platform = t.python.rsplit('.', 1)[0], t.platforms[0]
        self.marker, self._hooks = marker(self.platform), None

    @property
    def hooks(self):
        "The app's hooks module, or an empty namespace."
        if self._hooks is None:
            f = self.t.hooks and self.root/self.t.hooks
            if f and f.is_file():
                s = importlib.util.spec_from_file_location(f'{self.t.package}_kavacha_hooks', f)
                self._hooks = importlib.util.module_from_spec(s); s.loader.exec_module(self._hooks)
            else: self._hooks = SimpleNamespace()
        return self._hooks
    def hook(self, name): return getattr(self.hooks, name, None)
    def spec(self): return self.t.spec_env(self.platform.split('-')[0])

The pinned uv is the one the app ships: `<PREFIX>UV`, else the app's build folder. A uv of another version is refused, so a lock is never made or tested with a uv the app does not have.

In [ ]:
#| export
def _pinned_version(self): return (self.root/self.t.uv_version).read_text().strip()

def _pinned_uv(self):
    "The uv the app ships, refused when it is not the pinned version."
    uv = Path(os.environ.get(f'{self.t.env_prefix}UV') or self.root/self.t.build_dir/'uv'/'uv')
    if not uv.is_file(): raise SystemExit(f'no uv at {uv}; run kavacha ship mac --uv-only')
    if (v := uv_version(uv)) != self.pinned_version(): raise SystemExit(f'{uv} is uv {v}, not the pinned {self.pinned_version()}')
    return uv
Lock.pinned_version, Lock.pinned_uv = _pinned_version, _pinned_uv

### The child environment

`child_env` is what the launcher's own uv runs get, and `server_env` is what the server gets: the person's environment, less the dropped names, with the tool folders on `PATH`. `server_env_old` is what launcher 1.0.0 gave its server; a newer lock must still run under it.

In [ ]:
#| export
def _kept(s, base):
    base = dict(os.environ if base is None else base)
    return {k: v for k, v in base.items() if k not in s['drop'] and not k.startswith(tuple(s['drop_prefixes']))}

def _child_env(self, base=None, **vals):
    "The environment uv gets from the launcher: dropped keys, uv settings, `vals` filled in."
    s = self.spec()
    return s['set'] | _kept(s, base) | {k: v.format(**vals) for k, v in s['uv'].items()}

def _server_env(self, base=None):
    "The server's environment, as the launcher's `serverEnv`."
    s = self.spec(); e = _kept(s, base)
    path = [os.path.expanduser(d) for d in s['tool_dirs']] + e.get('PATH', '').split(os.pathsep)
    return e | s['server'] | dict(PATH=os.pathsep.join(dict.fromkeys(filter(None, path))))

def _server_env_old(self, env, base=None, **vals):
    "The server's environment from launcher 1.0.0: uv's build settings, VIRTUAL_ENV and the env's bin first."
    e = self.server_env(base)
    return e | self.child_env(base, env=env, **vals) | dict(PYTHONNOUSERSITE='1', VIRTUAL_ENV=str(env), PATH=os.pathsep.join([f'{env}/bin', e['PATH']]))

def _sync_args(self, lock, extras=(), inexact=False):
    "The build command; `inexact` adds an extra to an env without removing what it has."
    s = self.spec()
    return [a.format(lock=lock) for a in s['sync']] + (s['inexact'] if inexact else []) + [a.format(name=x) for x in extras for a in s['extra']]
Lock.child_env, Lock.server_env, Lock.server_env_old, Lock.sync_args = _child_env, _server_env, _server_env_old, _sync_args

In [ ]:
lk = Lock(Thin('Demo', repo='me/demo'), d)
e = lk.child_env(dict(PATH='/usr/bin', VIRTUAL_ENV='/v', UV_INDEX_URL='x', DEMO_HOST_TOKEN='t'), cache='/c', python='/py', python_version='3.13.18', env='/e')
test_eq(('VIRTUAL_ENV' in e, 'UV_INDEX_URL' in e, 'DEMO_HOST_TOKEN' in e, e['UV_PROJECT_ENVIRONMENT']), (False, False, False, '/e'))
test_eq(lk.sync_args('/l', ['a'], inexact=True)[-3:], ['--inexact', '--extra', 'a'])

### Writing the pinned project

`pyproject` writes the pinned project for app version `v`. The app's package comes from a wheel URL on its GitHub release (or a local wheel, for a lock that cannot be published). The cooldown is `exclude-newer`, `Thin.cooldown` days before the lock; the owner's packages are exempt. A pinned wheel (`Thin.wheels`) comes by URL.

In [ ]:
#| export
def _owners(self):
    "The packages the owner publishes, from the metadata installed here; they skip the cooldown."
    import importlib.metadata as md
    by = lambda d: ((d.metadata.get('Author-email') or '') + (d.metadata.get('Author') or '')).lower()
    return sorted({d.metadata['Name'].lower() for d in md.distributions() if any(o.lower() in by(d) for o in self.t.owners)} - {self.t.package}) if self.t.owners else []

def _extra_reqs(self):
    "Each extra's requirements, from the app's `extra_reqs` hook."
    f = self.hook('extra_reqs')
    return f() if f else {x: [] for x in self.t.extras}

def _constraints(self):
    f = self.root/'pyproject.toml'
    return tomllib.loads(f.read_text()).get('tool', {}).get('uv', {}).get('constraint-dependencies', []) if f.is_file() else []

def _pyproject(self, v, src, extras, owned=(), at=None, pins=()):
    "The pinned project's pyproject.toml. `src` is a wheel URL, or a local wheel path."
    at, t = at or now(), self.t
    q = lambda xs: '[' + ', '.join(json.dumps(x) for x in xs) + ']'
    s = f'{{ url = {json.dumps(src)} }}' if '://' in src else f'{{ path = {json.dumps(str(src))} }}'
    pkgs = ', '.join(f'{json.dumps(o)} = {json.dumps(iso(at))}' for o in owned)
    envs = ', '.join(json.dumps(marker(p)) for p in t.platforms)
    wheels = ''.join(f'{n} = {{ url = "{w["url"]}" }}\n' for n, w in t.wheels.items())
    return f'''[project]
name = "{t.lock_project}"
version = "0"
requires-python = "=={self.minor}.*"
dependencies = ["{t.package}=={v}"]

[project.optional-dependencies]
{chr(10).join(f'{k} = {q(extras[k])}' for k in t.extras)}

[tool.uv]
package = false
environments = [{envs}]
required-environments = [{envs}]
exclude-newer = "{iso(at - timedelta(days=t.cooldown))}"
exclude-newer-package = {{ {pkgs} }}
constraint-dependencies = {q([*self.constraints(), *pins])}

[tool.uv.sources]
{t.package} = {s}
{wheels}'''
Lock.owners, Lock.extra_reqs, Lock.constraints, Lock.pyproject = _owners, _extra_reqs, _constraints, _pyproject

In [ ]:
lk = Lock(Thin('Demo', repo='me/demo', extras=['x'], wheels={'big': dict(version='1', url='https://w/big-1-py3-none-any.whl', sha256='00')}), tempfile.mkdtemp())
p = tomllib.loads(lk.pyproject('1.0', 'https://w/demo.whl', {'x': ['big==1']}, ['mine'], datetime(2026, 11, 6, tzinfo=timezone.utc)))
test_eq((p['project']['dependencies'], p['tool']['uv']['exclude-newer'], p['tool']['uv']['sources']['big']), (['demo==1.0'], '2026-10-30T00:00:00Z', {'url': 'https://w/big-1-py3-none-any.whl'}))

### Checking a lock

`check` refuses a lock the app cannot install: another index, an editable, git or path source, a package with no fitting wheel, a pinned wheel in the base env or with another hash, and no required environment. It gives every reason, so one run lists them all.

In [ ]:
#| export
def packages(d): return tomllib.loads((Path(d)/'uv.lock').read_text()).get('package', [])

def _closure(pkgs, roots):
    by, seen, todo = {p['name']: p for p in pkgs}, set(), list(roots)
    while todo:
        if (n := todo.pop()) in seen or n not in by: continue
        seen.add(n)
        p = by[n]
        todo += [x['name'] for x in p.get('dependencies', [])]
        for x in p.get('dependencies', []): todo += [y['name'] for e in x.get('extra', []) for y in p.get('optional-dependencies', {}).get(e, [])]
    return seen

def _wname(w): return w.get('filename') or (w.get('url') or w.get('path') or '').rsplit('/', 1)[-1]

def _check(self, d, local=False):
    "Every reason the app cannot install the lock in `d`; [] when it can."
    t, pkgs, why = self.t, packages(d), []
    proj = tomllib.loads((Path(d)/'pyproject.toml').read_text())
    want = [marker(p) for p in t.platforms]
    if proj.get('tool', {}).get('uv', {}).get('required-environments') != want: why.append(f'required-environments is not {want}')
    if not tomllib.loads((Path(d)/'uv.lock').read_text()).get('options', {}).get('exclude-newer'): why.append('the lock records no exclude-newer cooldown')
    root = next((p for p in pkgs if p['name'] == t.lock_project), None)
    if root is None: return why + [f'the lock has no {t.lock_project} project']
    if missing := [x for x in t.extras if x not in root.get('optional-dependencies', {})]: why.append(f'extras missing: {missing}')
    for p in pkgs:
        n, src = p['name'], p.get('source', {})
        if n == t.lock_project: continue
        if (k := next(iter(src), None)) in ('editable', 'directory', 'git', 'virtual'): why.append(f'{n} comes from a {k} source')
        elif k == 'path' and not (n == t.package and local): why.append(f'{n} comes from a local path')
        elif k == 'url' and n in t.wheels:
            w = t.wheels[n]
            if src['url'] != w['url'] or not any(x.get('hash') == f"sha256:{w['sha256']}" for x in p.get('wheels', [])): why.append(f"{n} is not the pinned wheel {w['url']}")
        elif k == 'url' and n == t.package:
            if not src['url'].startswith(f'{self.dl}/'): why.append(f"{n} comes from {src['url']}, not a release of {self.repo}")
        elif k == 'url': why.append(f'{n} comes from a URL')
        elif k == 'registry' and src['registry'].rstrip('/') != PYPI: why.append(f"{n} comes from {src['registry']}")
        if not any(wheel_fits(_wname(w), t.python, pl) for w in p.get('wheels', []) for pl in t.platforms) and n not in t.source_only:
            why.append(f'{n} {p.get("version")} has no wheel for CPython {self.minor} on {", ".join(t.platforms)}')
    base = _closure(pkgs, [x['name'] for x in root.get('dependencies', [])])
    why += [f'{n} is in the base environment; it belongs in an extra only' for n in t.wheels if n in base]
    return why
Lock.check = _check

`make` writes the pinned project, locks it twice with the pinned uv (the second time with each extra pinned `==` to what the first chose), checks it, and prints its hash. `UV_NO_CONFIG=1` drops the settings in `[tool.uv]`, so the cooldown goes on the command line too.

In [ ]:
#| export
def _pin_extras(self, text, d):
    "The extras with every requirement pinned `==` to the version the lock chose."
    ver = {p['name']: p['version'] for p in packages(d)}
    def pin(r):
        name = re.split(r'[\[<>=!~; ]', r, maxsplit=1)[0]
        extra = re.match(r'[^\[]*(\[[^\]]*\])', r)
        mark = r.split(';', 1)[1] if ';' in r else ''
        key = name.lower().replace('_', '-')
        return f"{name}{extra.group(1) if extra else ''}=={ver[key]}" + (f'; {mark.strip()}' if mark else '') if key in ver else r
    names, out = '|'.join(map(re.escape, self.t.extras)) or '(?!)', []
    for line in text.splitlines():
        if (m := re.match(rf'^({names}) = (\[.*\])$', line)): line = f'{m[1]} = ' + json.dumps([pin(r) for r in json.loads(m[2])])
        out.append(line)
    return '\n'.join(out) + '\n'

def cooldown_args(d):
    "The pinned project's cooldown as uv flags, which UV_NO_CONFIG does not drop."
    u = tomllib.loads((Path(d)/'pyproject.toml').read_text())['tool']['uv']
    return ['--exclude-newer', u['exclude-newer'], *[a for k, v in u.get('exclude-newer-package', {}).items() for a in ('--exclude-newer-package', f'{k}={v}')]]

def _release_wheel(self, v, wait=600, every=20):
    "The URL and sha256 of the app's wheel on GitHub release `v`, waited for up to `wait` seconds."
    name, end = f'{self.t.package.replace("-", "_")}-{v}-py3-none-any.whl', time.monotonic() + wait
    while True:
        try:
            r = json.loads(urllib.request.urlopen(f'https://api.github.com/repos/{self.repo}/releases/tags/{v}', timeout=10).read())
            if a := next((a for a in r.get('assets', []) if a['name'] == name and a.get('digest')), None): return a['browser_download_url'], a['digest'].removeprefix('sha256:')
        except Exception: pass
        if time.monotonic() > end: raise SystemExit(f'no {name} on release {v}; run the lock wheel step for {v} first')
        print(f'waiting for {name} on release {v}'); time.sleep(every)

def _make(self, v, out=None, wheel=None, owned=None, pins=()):
    "Write, lock, pin and check the pinned project for app version `v`; print its hash."
    uv, out = self.pinned_uv(), Path(out or self.lock)
    out.mkdir(parents=True, exist_ok=True)
    src = str(Path(wheel).resolve()) if wheel else None
    if src is None: src, digest = self.release_wheel(v)
    (out/'.python-version').write_text(self.t.python + '\n')
    text = self.pyproject(v, src, self.extra_reqs(), self.owners() if owned is None else owned, pins=pins)
    (out/'pyproject.toml').write_text(text)
    cache = Path(os.environ.get('UV_CACHE_DIR') or self.user_cache())
    env = self.child_env(cache=cache, python=cache.parent/'python-lock', python_version=self.t.python, env=cache.parent/'lock-venv')
    env.pop('UV_FROZEN', None)
    run = lambda: subprocess.run([uv, 'lock', '--project', out, *cooldown_args(out)], env=env, check=True)
    run()
    (out/'pyproject.toml').write_text(self.pin_extras(text, out))
    run()
    if not wheel and not any(w.get('hash') == f'sha256:{digest}' for p in packages(out) if p['name'] == self.t.package for w in p.get('wheels', [])):
        raise SystemExit(f'the locked {self.t.package} wheel does not match the sha256 GitHub gives: {digest}')
    if why := self.check(out, local=bool(wheel)): raise SystemExit('the lock is refused:\n  ' + '\n  '.join(why))
    h = lock_hash(out, uv_version(uv))
    print(f'lock hash {h}' + ('  (local wheel: test only, cannot be published)' if wheel else ''))
    return h

def _user_cache(self): return Path.home()/'Library'/'Caches'/self.t.name/'uv'
Lock.pin_extras, Lock.release_wheel, Lock.make, Lock.user_cache = _pin_extras, _release_wheel, _make, _user_cache

### The manifest

`manifest` is what the launcher checks before it installs a lock. The app's own version is under the key of its package name, such as `leela`.

In [ ]:
#| export
def _download_bytes(self, d):
    "What a first install downloads: one fitting wheel per package of the base env, no extras."
    pkgs = packages(d)
    root = next((p for p in pkgs if p['name'] == self.t.lock_project), {})
    base = _closure(pkgs, [x['name'] for x in root.get('dependencies', [])])
    return sum(max((w.get('size') or 0 for w in p.get('wheels', []) if wheel_fits(_wname(w), self.t.python, self.platform)), default=0) for p in pkgs if p['name'] in base)

def _app_version(self, d): return next((p['version'] for p in packages(d) if p['name'] == self.t.package), None)

def _manifest(self, d, n, uv):
    "The manifest of lock `n` in `d`."
    return {'n': n, 'hash': lock_hash(d, uv), 'files': {f: sha256(Path(d)/f) for f in FILES}, self.t.package: self.app_version(d), 'python': self.t.python,
            'uv_min': uv, 'launcher_api_min': self.t.api, 'platform': self.platform, 'macos_min': self.t.macos_min, 'download_bytes': self.download_bytes(d)}
Lock.download_bytes, Lock.app_version, Lock.manifest = _download_bytes, _app_version, _manifest

### Signing

The keys are SSH Ed25519 keys in `~/.ssh`, named by `Thin.key_names`: main and emergency. Signing is SSHSIG with the namespace `Thin.sig_ns`, through the SSH agent when it holds the key. The tools check a signature with `ssh-keygen -Y verify`; the launcher checks SSHSIG itself.

In [ ]:
#| export
def _key_paths(self, home=None): return [Path(home or Path.home())/'.ssh'/k for k in self.t.key_names]

def _sign(self, path, key):
    "Sign `path` with `key`; the signature is `<path>.sig`."
    Path(f'{path}.sig').unlink(missing_ok=True)
    pub = Path(f'{key}.pub')
    use = pub if pub.is_file() and pub.read_text().split()[1] in sh('ssh-add', '-L', check=False) else key
    sh('ssh-keygen', '-Y', 'sign', '-n', self.t.sig_ns, '-f', use, path)
    return Path(f'{path}.sig')

def _allowed(self, keys_pub):
    "An allowed_signers file for `ssh-keygen -Y verify`, from the public keys the app embeds."
    ns = self.t.sig_ns
    lines = [f'{ns} namespaces="{ns}" ' + ' '.join(l.split()[:2]) for l in Path(keys_pub).read_text().splitlines() if l.strip().startswith('ssh-ed25519')]
    if not lines: raise SystemExit(f'no ssh-ed25519 key in {keys_pub}')
    (f := Path(tempfile.mkdtemp())/'allowed_signers').write_text('\n'.join(lines) + '\n')
    return f

def _verify_sig(self, data, sig, keys_pub):
    "Whether `sig` is a good signature over the file `data` by a key in `keys_pub`."
    with open(data, 'rb') as f:
        return subprocess.run(['ssh-keygen', '-Y', 'verify', '-f', self.allowed(keys_pub), '-I', self.t.sig_ns, '-n', self.t.sig_ns, '-s', sig], stdin=f, capture_output=True).returncode == 0

def _check_lock_dir(self, d, keys_pub, highest=-1):
    "Check a lock as the launcher does: signature, file hashes, lock hash, platform. The manifest, or SystemExit."
    d = Path(d)
    if not self.verify_sig(d/'manifest.json', d/'manifest.sig', keys_pub): raise SystemExit(f'{d}/manifest.sig is not a good {self.t.sig_ns} signature')
    m = json.loads((d/'manifest.json').read_text())
    for f in FILES:
        if m['files'].get(f) != sha256(d/f): raise SystemExit(f'{f} does not match the manifest')
    if lock_hash(d, m['uv_min']) != m['hash']: raise SystemExit('the lock hash does not match the manifest')
    if m['platform'] not in self.t.platforms: raise SystemExit(f"the lock is for {m['platform']}")
    if m['n'] <= highest: raise SystemExit(f"lock {m['n']} is not newer than {highest}")
    return m

def _lock_key(self, home=None, keys_out=None, passphrase=True, agent=True):
    "Make the main and emergency keys in ~/.ssh, add them to the agent with the passphrase in the keychain, print the public keys."
    paths = self.key_paths(home)
    if have := [p for p in paths if p.exists() or Path(f'{p}.pub').exists()]: raise SystemExit(f'refusing to overwrite {", ".join(map(str, have))}')
    paths[0].parent.mkdir(mode=0o700, exist_ok=True)
    for p, who in zip(paths, ('main', 'emergency')):
        cmd = ['ssh-keygen', '-t', 'ed25519', '-a', '100', '-f', str(p), '-C', f'{self.t.sig_ns}-{who}'] + ([] if passphrase else ['-N', ''])
        if subprocess.run(cmd).returncode: raise SystemExit(f'ssh-keygen did not make {p}')
        if agent and subprocess.run(['ssh-add', '--apple-use-keychain', str(p)]).returncode: raise SystemExit(f'ssh-add did not take {p}')
    pubs = ''.join(Path(f'{p}.pub').read_text() for p in paths)
    print(f'public keys to embed ({self.t.keys}):\n{pubs}')
    if keys_out and not Path(keys_out).exists(): Path(keys_out).write_text(pubs); print(f'wrote {keys_out}; commit it')
    return pubs
Lock.key_paths, Lock.sign, Lock.allowed, Lock.verify_sig, Lock.check_lock_dir, Lock.lock_key = _key_paths, _sign, _allowed, _verify_sig, _check_lock_dir, _lock_key

### Fixtures and dev locks

`fixtures` writes SSHSIG fixtures with throwaway keys, under a temporary `HOME`, so no real key or agent is ever used. `dev_lock` signs a lock with a throwaway key, for a local app build that is not for release.

In [ ]:
#| export
def fixtures(out, ns='app-lock', platform='darwin-arm64'):
    "SSHSIG fixtures made by ssh-keygen with throwaway keys under a temporary HOME."
    out, tmp = Path(out), Path(tempfile.mkdtemp())
    out.mkdir(parents=True, exist_ok=True)
    env = dict(PATH='/usr/bin:/bin', HOME=str(tmp))
    for k in ('main', 'emergency', 'stranger'): sh('/usr/bin/ssh-keygen', '-q', '-t', 'ed25519', '-N', '', '-C', f'fixture-{k}', '-f', tmp/k, env=env)
    (msg := out/'message.json').write_text(dumps(dict(n=7, hash='ab'*32, platform=platform)))
    for k, n, name in (('main', ns, 'main'), ('emergency', ns, 'emergency'), ('stranger', ns, 'stranger'), ('main', 'other-ns', 'wrongns')):
        sh('/usr/bin/ssh-keygen', '-Y', 'sign', '-n', n, '-f', tmp/k, msg, env=env)
        Path(f'{msg}.sig').rename(out/f'{name}.sig')
    (out/'keys.pub').write_text(''.join((tmp/f'{k}.pub').read_text() for k in ('main', 'emergency')))
    shutil.copy(tmp/'main', out/'main.key')
    shutil.rmtree(tmp)
    return out

def _dev_lock(self, out, src=None, n=0):
    "A lock signed with a throwaway key: the pinned project in `src`, manifest `n`, keys.pub with the throwaway key."
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    for f in FILES: shutil.copy(Path(src or self.lock)/f, out/f)
    k = Path(tempfile.mkdtemp())/'dev'
    sh('/usr/bin/ssh-keygen', '-q', '-t', 'ed25519', '-N', '', '-C', f'{self.t.sig_ns}-dev', '-f', k)
    (out/'manifest.json').write_text(dumps(self.manifest(out, n, self.pinned_version())))
    sh('/usr/bin/ssh-keygen', '-Y', 'sign', '-n', self.t.sig_ns, '-f', k, out/'manifest.json')
    Path(f"{out/'manifest.json'}.sig").rename(out/'manifest.sig')
    shutil.copy(f'{k}.pub', out/'keys.pub')
    shutil.rmtree(k.parent)
    print(f'dev lock in {out} (throwaway key; not for release)')
    return out
Lock.dev_lock = _dev_lock

In [ ]:
f = fixtures(Path(tempfile.mkdtemp()))
lk = Lock(Thin('App'), d)
test_eq(lk.verify_sig(f/'message.json', f/'main.sig', f/'keys.pub'), True)
test_eq([lk.verify_sig(f/'message.json', f/f'{k}.sig', f/'keys.pub') for k in ('emergency', 'stranger', 'wrongns')], [True, False, False])

### The smoke test

`lock_test` builds the env as the launcher does, in a temp folder, with the same uv, child environment and command. Then `smoke` checks it, and a pass writes a test record for this lock hash. `Smoke` is what a hook gets: the env, the temp folders, and `step`, `serve` and the HTTP helpers.

In [ ]:
#| export
def live_ports(name, home=None):
    "The ports of the app the person runs, from its real serving.json: no test may call them."
    f = Path(home or Path.home())/'.config'/name.lower()/'serving.json'
    try: d = json.loads(f.read_text()); os.kill(int(d['pid']), 0)
    except (OSError, ValueError, KeyError, TypeError): return set()
    return {int(d['port']), *map(int, d.get('ports') or ())}

def free_port():
    with socket.socket() as s: s.bind(('127.0.0.1', 0)); return s.getsockname()[1]

class FakeHost:
    "The launcher's control server, enough for a smoke test: every question gets Yes."
    def __init__(self, token):
        from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
        host, self.calls = self, []
        class H(BaseHTTPRequestHandler):
            def _do(self):
                n = int(self.headers.get('Content-Length') or 0)
                host.calls.append((self.command, self.path, self.rfile.read(n) if n else b''))
                ok = self.headers.get('Authorization') == f'Bearer {token}'
                body = json.dumps({'answer': True} if self.path == '/confirm' else {'ok': True}).encode() if ok else b'{"ok":false}'
                self.send_response(200 if ok else 401); self.send_header('Content-Length', str(len(body))); self.end_headers(); self.wfile.write(body)
            do_GET = do_POST = _do
            def log_message(self, *a): pass
        self.srv = ThreadingHTTPServer(('127.0.0.1', 0), H)
        self.url = f'http://127.0.0.1:{self.srv.server_address[1]}'
        threading.Thread(target=self.srv.serve_forever, daemon=True).start()
    def close(self): self.srv.shutdown()

`Smoke.get` and `Smoke.post` never reach the app the person runs: a port in its `serving.json` is refused. Proxies are never used.

In [ ]:
#| export
class Smoke:
    "What a smoke check needs: the env, temp folders, results, and the helpers to start and call a server."
    def __init__(self, lk, env_dir, results, extras=(), wait=180):
        self.lk, self.t, self.env_dir, self.results, self.extras, self.wait = lk, lk.t, Path(env_dir), results, extras, wait
        self.py, self.uv = self.env_dir/'bin'/'python', lk.pinned_uv()
        self.tmp = Path(tempfile.mkdtemp(prefix=f'{lk.t.package}-smoke-')).resolve()
        for n in ('cfg', 'data', 'work'): (self.tmp/n).mkdir()
        self.server_extra = {}

    def step(self, name, f):
        "Run `f` as one step of the record; whether it passed."
        t = time.monotonic()
        try: f(); self.results.append(dict(step=name, ok=True, secs=round(time.monotonic()-t, 1)))
        except Exception as e: self.results.append(dict(step=name, ok=False, secs=round(time.monotonic()-t, 1), error=str(e)[-1500:])); return False
        return True

    def get(self, url, data=None, headers=None, timeout=5, raw=False):
        if urllib.parse.urlsplit(url).port in live_ports(self.t.name): raise RuntimeError(f'{url} is the {self.t.name} you are running; tests use their own server')
        req = urllib.request.Request(url, data=data, headers=headers or {}, method='POST' if data is not None else 'GET')
        with urllib.request.build_opener(urllib.request.ProxyHandler({})).open(req, timeout=timeout) as r: return r.read() if raw else json.loads(r.read())

    def post(self, url, body, timeout=180):
        "A JSON POST's status and answer, a refusal included."
        if urllib.parse.urlsplit(url).port in live_ports(self.t.name): raise RuntimeError(f'{url} is the {self.t.name} you are running')
        req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={'Content-Type': 'application/json'}, method='POST')
        try:
            with urllib.request.build_opener(urllib.request.ProxyHandler({})).open(req, timeout=timeout) as r: return r.status, r.read().decode()
        except urllib.error.HTTPError as e: return e.code, e.read().decode()

    def server_env(self, base=None, **kw):
        "The server's environment with this test's own config and data folders."
        return self.lk.server_env(base) | dict(XDG_CONFIG_HOME=str(self.tmp/'cfg'), XDG_DATA_HOME=str(self.tmp/'data')) | self.server_extra | kw

`Smoke.serve` starts the server from an env as the launcher does: the host variables in its environment, the token as one stdin line, never in argv, the environment, a file or a log. It waits for the health route, runs `checks(base, port)`, then stops it. A server must stop within 5 s of SIGTERM, with the streams the app's `hold_open(s, base, port)` hook names held open, as a window holds them.

In [ ]:
#| export
def _start(self, env_dir, env, name, port, token):
    host = FakeHost(token)
    env = env | host_env(self.t.env_prefix, host.url)
    args = [a.format(port=port) for a in self.lk.spec()['server_args']]
    log = self.tmp/f'server-{name}.log'
    with open(log, 'wb') as f: srv = subprocess.Popen([Path(env_dir)/'bin'/self.t.server, *args], env=env, cwd=self.tmp, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.PIPE)
    srv.stdin.write(f'{token}\n'.encode()); srv.stdin.close()
    return host, srv, log

def _healthy(self, srv, base, log, wait):
    end = time.monotonic() + wait
    while time.monotonic() < end:
        if srv.poll() is not None: raise RuntimeError(f'the server stopped with {srv.returncode}: ' + log.read_text()[-1500:])
        try:
            if self.get(f'{base}{self.t.health}', timeout=2).get('status') == 'ok': return
        except Exception: time.sleep(.5)
    raise TimeoutError(f'no {self.t.health} in {wait} s: ' + log.read_text()[-1500:])

def _serve(self, name, env, checks=None, env_dir=None, quit=True):
    "The server in host mode under `env`, its health, `checks(base, port)`, then a quit; whether all passed."
    port, token, host = free_port(), os.urandom(16).hex(), None
    try:
        host, srv, log = self._start(env_dir or self.env_dir, env, name.strip(' ()').replace(' ', '-') or 'main', port, token)
        base = f'http://127.0.0.1:{port}'
        good = self.step(f'server {self.t.health}{name}', lambda: self._healthy(srv, base, log, self.wait))
        if good and checks: good = checks(base, port) is not False
        def stop():
            held = (self.lk.hook('hold_open') or (lambda s, b, p: []))(self, base, port)
            socks = [socket.create_connection(('127.0.0.1', p)) for p, _ in held]
            for s, (p, path) in zip(socks, held): s.sendall(f'GET {path} HTTP/1.1\r\nHost: 127.0.0.1:{p}\r\n\r\n'.encode())
            if socks: time.sleep(1)
            t = time.monotonic(); srv.terminate()
            try: srv.wait(15)
            except subprocess.TimeoutExpired: srv.kill(); raise RuntimeError(f'SIGTERM did not stop the server in 15 s with {len(socks)} page streams open')
            finally:
                for s in socks: s.close()
            if (secs := time.monotonic() - t) > 5: raise RuntimeError(f'the server took {secs:.1f} s to stop')
        done = self.step(f'quit with a page open{name}', stop) if quit else True
        if srv.poll() is None: srv.kill()
        return done and good
    finally:
        if host: host.close()

def _healthy_env(self, env_dir, wait=120):
    "Whether the server from `env_dir` answers its health route in host mode."
    port, token, host = free_port(), os.urandom(16).hex(), None
    try:
        host, srv, log = self._start(env_dir, self.server_env(), f'health-{Path(env_dir).name}', port, token)
        try: self._healthy(srv, f'http://127.0.0.1:{port}', log, wait); return True
        except Exception: return False
        finally:
            srv.terminate()
            try: srv.wait(15)
            except subprocess.TimeoutExpired: srv.kill()
    finally:
        if host: host.close()
Smoke._start, Smoke._healthy, Smoke.serve, Smoke.healthy = _start, _healthy, _serve, _healthy_env

`Lock.smoke` runs the steps every thin app needs, then the app's hooks: `uv pip check`, the imports, the first server with the app's `checks`, then the app's `smoke(s)`, then each extra's check (`extra_check(x)` gives Python arguments, else `import x`).

In [ ]:
#| export
def _smoke(self, env_dir, results, extras=(), wait=180):
    "Check an installed env; whether every step passed."
    s, h = Smoke(self, env_dir, results, extras, wait), self.hook
    if f := h('prepare'): f(s)
    ok = s.step('pip check', lambda: sh(s.uv, 'pip', 'check', '--python', s.py))
    ok &= s.step('imports', lambda: sh(s.py, '-I', '-c', 'import ' + ', '.join([self.t.package, *self.t.imports])))
    if f := h('prepare_steps'): ok &= f(s) is not False
    ok &= s.serve('', s.server_env(), (lambda b, p: h('checks')(s, b, p)) if h('checks') else None)
    if f := h('smoke'): ok &= f(s) is not False
    for x in extras:
        args = (h('extra_check') or (lambda x: None))(x) or ['-c', f'import {x}']
        ok &= s.step(f'extra {x}', lambda args=args: sh(s.py, *args, env=s.server_env()))
    shutil.rmtree(s.tmp, ignore_errors=True)
    return ok
Lock.smoke = _smoke

### The env from a copy

The launcher builds a new lock's env from an APFS clone of the env in use. `seed` does the same here: a clone with the old path rewritten in `bin/` and `pyvenv.cfg`, and no bytecode. `incremental` then adds one extra with `uv sync` and checks that the copy installs only that, works, and names only itself.

In [ ]:
#| export
def seed(a, b):
    "`b` as a clone of env `a`, with `a`'s path rewritten in bin/ and pyvenv.cfg, and no bytecode."
    sh('cp', '-c', '-R', a, b)
    for c in list(Path(b).rglob('__pycache__')): shutil.rmtree(c, ignore_errors=True)
    old, new = str(a).encode(), str(b).encode()
    for f in [Path(b)/'pyvenv.cfg', *(Path(b)/'bin').iterdir()]:
        if f.is_symlink() or not f.is_file() or f.stat().st_size > 2_000_000: continue
        if old in (d := f.read_bytes()): f.write_bytes(d.replace(old, new))

def _digest(env, server):
    return hashlib.sha256(b''.join((env/f).read_bytes() for f in ('pyvenv.cfg', f'bin/{server}')) + '\n'.join(sorted(p.name for p in (env/'bin').iterdir())).encode()).hexdigest()

def _incremental(self, env_a, lock, cache, tmp, uv, results):
    "The next env from a copy of this one: `uv sync` adds one extra, and the copy works and names only itself."
    t, b, srv = time.monotonic(), tmp/'next', self.t.server
    try:
        before = _digest(env_a, srv)
        seed(env_a, b)
        env = self.child_env(cache=cache, python=tmp/'python', python_version=self.t.python, env=b)
        p = subprocess.run([str(uv), *self.sync_args(lock, [self.t.seed_extra] if self.t.seed_extra else [])], env=env, capture_output=True, text=True, timeout=600)
        if p.returncode: raise RuntimeError(f'uv sync on the copy: {(p.stderr or p.stdout)[-800:]}')
        n = int(m.group(1)) if (m := re.search(r'Installed (\d+) packages?', p.stderr + p.stdout)) else 0
        bad = [] if 0 < n < 20 or not self.t.seed_extra else [f'the copy installed {n} packages, not only the extra']
        bad += [str(f) for f in b.rglob('*') if f.is_file() and not f.is_symlink() and f.stat().st_size < 2_000_000 and str(env_a).encode() in f.read_bytes()]
        bad += [] if (b/'bin'/srv).read_text().startswith(f'#!{b}/bin/python') else [f'bin/{srv} does not run the copy\'s python']
        bad += [] if _digest(env_a, srv) == before else ['the env in use changed']
        sh(b/'bin'/'python', '-I', '-c', 'import ' + ', '.join([self.t.package, *self.t.imports]))
        s = Smoke(self, b, [], wait=120)
        if f := self.hook('prepare'): f(s)
        if f := self.hook('copy_checks'): bad += f(s, b) or []
        if not s.healthy(b): bad.append(f'the server from the copy did not answer {self.t.health}')
        shutil.rmtree(s.tmp, ignore_errors=True)
        if bad: raise RuntimeError('; '.join(bad[:8]))
        results.append(dict(step=f'env from a copy ({n} installed)', ok=True, secs=round(time.monotonic()-t, 1)))
        return True
    except Exception as e:
        results.append(dict(step='env from a copy', ok=False, secs=round(time.monotonic()-t, 1), error=str(e)[-1500:]))
        return False
Lock.incremental = _incremental

`test_code_hash` covers the harness (this module), the app's hooks file and its child environment, so a record from other test code is refused at publish.

In [ ]:
#| export
def _test_code_hash(self):
    hk = self.t.hooks and self.root/self.t.hooks
    return hashlib.sha256(Path(__file__).read_bytes() + (hk.read_bytes() if hk and hk.is_file() else b'') + json.dumps(self.spec(), sort_keys=True).encode()).hexdigest()

def platform_mac():
    import platform
    return platform.mac_ver()[0]

def _lock_test(self, d=None, fresh=False, extras=False, record_dir=None, keep=False):
    "Build the env as the app does, in a temp folder, smoke it, and write the test record for this lock hash."
    uv, d = self.pinned_uv(), Path(d or self.lock)
    uvv = uv_version(uv); h = lock_hash(d, uvv)
    tmp = Path(tempfile.mkdtemp(prefix=f'{self.t.package}-lock-test-')).resolve()
    (lock := tmp/'lock').mkdir()
    for f in FILES: shutil.copy(d/f, lock/f)
    cache = tmp/'cache' if fresh else Path(os.environ.get(f'{self.t.env_prefix}LOCK_CACHE') or self.user_cache())
    env_dir, results, t0 = tmp/'env', [], time.monotonic()
    env = self.child_env(cache=cache, python=tmp/'python', python_version=self.t.python, env=env_dir)
    xs = tuple(self.t.extras) if extras else ()
    print(f'building {h[:16]} in {tmp} (cache {cache})')
    with open(tmp/'sync.log', 'wb') as log: rc = subprocess.run([uv, *self.sync_args(lock, xs)], env=env, stdout=log, stderr=subprocess.STDOUT).returncode
    results.append(dict(step='uv sync', ok=rc == 0, secs=round(time.monotonic()-t0, 1), **({} if rc == 0 else {'error': (tmp/'sync.log').read_text()[-1500:]})))
    ok = rc == 0 and self.smoke(env_dir, results, xs)
    if ok and not xs: ok &= self.incremental(env_dir, lock, cache, tmp, uv, results)
    size = sum(f.stat().st_size for f in env_dir.rglob('*') if f.is_file() and not f.is_symlink()) if env_dir.exists() else 0
    rec = {'hash': h, 'passed': ok, 'uv': uvv, 'python': self.t.python, 'macos': platform_mac(), 'launcher_api': self.t.api, 'test_code': self.test_code_hash(),
           'time': iso(now()), 'fresh': fresh, 'extras': list(xs), 'env_bytes': size, self.t.package: self.app_version(d), 'results': results}
    for r in results: print(f"  {'ok  ' if r['ok'] else 'FAIL'} {r['step']} ({r['secs']} s)" + (f"\n       {r['error'][-600:]}" if not r['ok'] else ''))
    out = Path(record_dir or d/'tested')
    if ok:
        out.mkdir(parents=True, exist_ok=True)
        (out/f'{h}.json').write_text(dumps(rec))
        print(f'passed; record {out}/{h}.json; env {size/2**20:.0f} MB')
    else: print('failed; no record written')
    if not keep: shutil.rmtree(tmp, ignore_errors=True)
    return rec
Lock.test_code_hash, Lock.lock_test = _test_code_hash, _lock_test

### The channel

Each lock is a release `lock-<n>`; the release `lock-channel` holds the signed pointers `channel.json` and `channel-test.json`. A pointer names its channel, so a test pointer served as stable is refused. `gh` runs with the keyring's sign-in, never a `GITHUB_TOKEN` from the environment.

In [ ]:
#| export
def fetch(url, dst=None, fresh=False, tries=1, wait=5):
    "The bytes at `url`, also written to `dst`. `fresh` gets past GitHub's cache of a replaced asset; `tries` waits out a new release's 404."
    if fresh: url += ('&' if '?' in url else '?') + f't={uuid.uuid4().hex}'
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=30) as r: b = r.read()
            break
        except urllib.error.HTTPError as e:
            if e.code != 404 or i == tries - 1: raise
            time.sleep(wait)
    if dst: Path(dst).write_bytes(b)
    return b

def pointer(n, manifest_sha, name, at=None):
    "A channel pointer; it names its channel."
    return dict(n=n, manifest_sha256=manifest_sha, published=iso(at or now()), channel=name)

def _gh(self, *a, check=True): return sh('gh', *a, '-R', self.repo, check=check, env={k: v for k, v in os.environ.items() if k != 'GITHUB_TOKEN'})

def _channel(self, name='channel', keys_pub=None):
    "The channel pointer `name`, checked; {} when none is published."
    tmp = Path(tempfile.mkdtemp())
    # a pointer is replaced in place, and GitHub serves the old one for minutes unless the address is new
    try: fetch(f'{self.dl}/lock-channel/{name}.json', tmp/'c.json', fresh=True); fetch(f'{self.dl}/lock-channel/{name}.sig', tmp/'c.sig', fresh=True)
    except urllib.error.HTTPError as e:
        if e.code == 404: return {}
        raise
    if not self.verify_sig(tmp/'c.json', tmp/'c.sig', keys_pub or self.keys_pub): raise SystemExit(f'{name}.json has a bad signature')
    c = json.loads((tmp/'c.json').read_text())
    if c.get('channel') != name: raise SystemExit(f"{name}.json is a pointer for {c.get('channel')}")
    return c

def _fetch_lock(self, n, into):
    into = Path(into); into.mkdir(parents=True, exist_ok=True)
    for f in ('manifest.json', 'manifest.sig', *FILES): fetch(f'{self.dl}/lock-{n}/{asset_name(f)}', into/f, tries=12)
    return into
Lock.gh, Lock.channel, Lock.fetch_lock = _gh, _channel, _fetch_lock

`refusals` is every reason `publish` refuses: no public keys, no signing key, a failed check, a tree that is not clean and pushed, no fresh passing record from this uv and this test code, and a lower app version than the last lock (unless `allow_downgrade`).

In [ ]:
#| export
def _record_for(self, h, d=None, at=None):
    "The passing test record for hash `h`, or why it cannot be used."
    p = Path(d or self.lock)/'tested'/f'{h}.json'
    if not p.is_file(): return None, f'no test record for {h[:16]}; run the lock test'
    r = json.loads(p.read_text())
    if not r.get('passed'): return None, 'the test record did not pass'
    age = (at or now()) - datetime.strptime(r['time'], '%Y-%m-%dT%H:%M:%SZ').replace(tzinfo=timezone.utc)
    if age > timedelta(days=self.t.record_days): return None, f'the test record is {age.days} days old; run the lock test again'
    if r.get('uv') != self.pinned_version(): return None, f"the record used uv {r.get('uv')}, not the pinned {self.pinned_version()}"
    if r.get('test_code') != self.test_code_hash(): return None, 'the lock test code or the child environment changed since the test; run the lock test again'
    return r, ''

def _tree_clean(self):
    "Why the git tree cannot publish, or ''."
    if sh('git', '-C', self.root, 'status', '--porcelain'): return 'the git tree has changes that are not committed'
    head, up = sh('git', '-C', self.root, 'rev-parse', 'HEAD'), sh('git', '-C', self.root, 'rev-parse', '@{u}', check=False)
    return '' if head == up else 'HEAD is not pushed to its upstream branch'

def _refusals(self, d=None, published=None, allow_downgrade=False, at=None, keys_pub=None):
    "Every reason `publish` refuses this lock."
    d, keys_pub, why = Path(d or self.lock), Path(keys_pub or self.keys_pub), []
    if not keys_pub.is_file(): why.append(f'no {keys_pub}; run the lock key step and commit the public keys')
    if not self.key_paths()[0].is_file(): why.append(f'no signing key {self.key_paths()[0]}; run the lock key step')
    why += self.check(d)
    if t := self.tree_clean(): why.append(t)
    r, msg = self.record_for(lock_hash(d, self.pinned_version()), d, at=at)
    if not r: why.append(msg)
    pk = self.t.package
    if published and published.get(pk) and not allow_downgrade:
        V = lambda v: tuple(int(x) for x in re.findall(r'\d+', v))
        if V(self.app_version(d)) < V(published[pk]): why.append(f"{pk} {self.app_version(d)} is lower than the published {published[pk]}; allow a downgrade to roll back")
    return why
Lock.record_for, Lock.tree_clean, Lock.refusals = _record_for, _tree_clean, _refusals

`publish` signs the lock as `lock-<n>`, checks what GitHub serves, then points the test channel at it. `promote` copies the test pointer to stable. `verify` checks a channel as the app does.

In [ ]:
#| export
def _publish(self, d=None, keys_pub=None, allow_downgrade=False):
    "Sign and publish this lock as lock-<n>, check it, then point the test channel at it."
    d, keys_pub = Path(d or self.lock), Path(keys_pub or self.keys_pub)
    if why := self.refusals(d, None, allow_downgrade, keys_pub=keys_pub): raise SystemExit('lock publish refuses:\n  ' + '\n  '.join(why))
    cur = self.channel('channel-test', keys_pub) or self.channel('channel', keys_pub)
    last = self.check_lock_dir(self.fetch_lock(cur['n'], Path(tempfile.mkdtemp())/'last'), keys_pub) if cur else None
    if why := self.refusals(d, last, allow_downgrade, keys_pub=keys_pub): raise SystemExit('lock publish refuses:\n  ' + '\n  '.join(why))
    n, key, v = (cur.get('n') or 0) + 1, self.key_paths()[0], self.pinned_version()
    out = Path(tempfile.mkdtemp(prefix=f'{self.t.package}-lock-{n}-'))
    for f in FILES: shutil.copy(d/f, out/asset_name(f))
    (out/'manifest.json').write_text(dumps(self.manifest(d, n, v)))
    self.sign(out/'manifest.json', key).rename(out/'manifest.sig')
    shutil.copy(d/'tested'/f'{lock_hash(d, v)}.json', out/'tested.json')
    self.gh('release', 'create', f'lock-{n}', '--latest=false', '--prerelease', '--title', f'Lock {n}', '--notes', f'Environment lock {n}: {self.t.package} {self.app_version(d)}.', *sorted(map(str, out.iterdir())))
    got = self.check_lock_dir(self.fetch_lock(n, out/'check'), keys_pub)
    (c := out/'channel-test.json').write_text(dumps(pointer(n, sha256(out/'manifest.json'), 'channel-test')))
    self.sign(c, key).rename(out/'channel-test.sig')
    self.gh('release', 'view', 'lock-channel', check=False) or self.gh('release', 'create', 'lock-channel', '--latest=false', '--prerelease', '--title', 'Lock channel', '--notes', 'Pointers to the current environment locks.')
    self.gh('release', 'upload', 'lock-channel', c, out/'channel-test.sig', '--clobber')
    if self.channel('channel-test', keys_pub).get('n') != n: raise SystemExit('the test channel does not point at the new lock')
    print(f"lock {n} ({self.t.package} {got.get(self.t.package)}) is on the test channel; {self.t.env_prefix}LOCK_CHANNEL=test points an app at it; then promote {n}")

def _promote(self, n, keys_pub=None):
    "Point the stable channel at lock `n`, which must be the one on the test channel."
    keys_pub = keys_pub or self.keys_pub
    t = self.channel('channel-test', keys_pub)
    if t.get('n') != n: raise SystemExit(f"the test channel points at lock {t.get('n')}, not {n}")
    out = Path(tempfile.mkdtemp())
    self.check_lock_dir(self.fetch_lock(n, out/'lock'), keys_pub)
    (c := out/'channel.json').write_text(dumps(pointer(n, t['manifest_sha256'], 'channel')))
    self.sign(c, self.key_paths()[0]).rename(out/'channel.sig')
    self.gh('release', 'upload', 'lock-channel', c, out/'channel.sig', '--clobber')
    self.verify(keys_pub=keys_pub)

def _verify(self, name='channel', keys_pub=None):
    "Check a published channel as the app does."
    keys_pub = keys_pub or self.keys_pub
    if not (c := self.channel(name, keys_pub)): raise SystemExit(f'no {name}.json is published')
    d = self.fetch_lock(c['n'], Path(tempfile.mkdtemp())/'lock')
    if sha256(d/'manifest.json') != c['manifest_sha256']: raise SystemExit('the manifest does not match the channel pointer')
    m = self.check_lock_dir(d, keys_pub)
    print(f"{name}: lock {m['n']}, {self.t.package} {m.get(self.t.package)}, hash {m['hash'][:16]}, {m['download_bytes']/2**20:.0f} MB to download: good")
    return m

def _fetch_channel(self, out, name='channel', keys_pub=None):
    "The lock a channel points at, checked, into `out`: the bootstrap lock of a release build."
    keys_pub = keys_pub or self.keys_pub
    if not (c := self.channel(name, keys_pub)): raise SystemExit(f'no {name}.json is published; publish and promote a lock first')
    d = self.fetch_lock(c['n'], out)
    if sha256(d/'manifest.json') != c['manifest_sha256']: raise SystemExit('the manifest does not match the channel pointer')
    shutil.copy(keys_pub, Path(out)/'keys.pub')
    m = self.check_lock_dir(d, keys_pub)
    print(f"bootstrap lock {m['n']} ({self.t.package} {m.get(self.t.package)}) in {out}")
    return d
Lock.publish, Lock.promote, Lock.verify, Lock.fetch_channel = _publish, _promote, _verify, _fetch_channel

`diff` lists what changed against the published lock. An owner package version with no git tag in `Thin.owner_org` is flagged. `wheel` builds the app's wheel from a tag in a clean worktree, and attaches it to that release.

In [ ]:
#| export
def _tagged(org, name, v):
    for t in (v, f'v{v}'):
        try: urllib.request.urlopen(f'https://api.github.com/repos/{org}/{name}/git/ref/tags/{t}', timeout=10); return True
        except Exception: pass
    return False

def _diff(self, d=None, keys_pub=None, old=None):
    "Packages added, removed and changed against the published lock."
    if old is None:
        c = self.channel('channel', keys_pub)
        old = packages(self.fetch_lock(c['n'], Path(tempfile.mkdtemp())/'lock')) if c else []
    a, b = {p['name']: p['version'] for p in old}, {p['name']: p['version'] for p in packages(d or self.lock)}
    rows = [f'+ {n} {b[n]}' for n in sorted(b.keys() - a.keys())] + [f'- {n} {a[n]}' for n in sorted(a.keys() - b.keys())]
    rows += [f'~ {n} {a[n]} -> {b[n]}' for n in sorted(a.keys() & b.keys()) if a[n] != b[n]]
    for n in sorted(set(self.owners()) & b.keys()):
        if (n not in a or a[n] != b[n]) and not _tagged(self.t.owner_org, n, b[n]): rows.append(f'! {n} {b[n]} has no git tag on {self.t.owner_org}/{n}')
    print('\n'.join(rows) or 'no change')
    return rows

def _wheel(self, v, upload=False):
    "Build the app's wheel from tag `v` in a clean worktree; with `upload`, attach it to release `v`."
    tmp = Path(tempfile.mkdtemp(prefix=f'{self.t.package}-wheel-'))
    sh('git', '-C', self.root, 'worktree', 'add', '--detach', tmp/'src', v)
    try:
        sh('uv', 'build', '--wheel', '--out-dir', tmp/'dist', tmp/'src')
        w = tmp/'dist'/f'{self.t.package.replace("-", "_")}-{v}-py3-none-any.whl'
        if not w.is_file(): raise SystemExit(f'the build made no {w.name}')
        if upload: self.gh('release', 'upload', v, w)
        print(f'{w}' + (f' is on release {v}' if upload else ''))
        return w
    finally: sh('git', '-C', self.root, 'worktree', 'remove', '--force', tmp/'src', check=False)
Lock.diff, Lock.wheel = _diff, _wheel

### Checking a built app

`verify_app` checks a built bundle without opening it: the plist, the launcher, the pinned uv, the bootstrap lock and its signature, the generated files, and the size gates of the spec. A bundle whose keys are not the committed keys is signed with a throwaway key, and it says so.

In [ ]:
#| export
def _verify_app(self, app, zip_=None):
    "Check a built app bundle; its facts, or SystemExit with every reason."
    import plistlib
    app, why, t = Path(app), [], self.t
    c = app/'Contents'
    p = plistlib.loads((c/'Info.plist').read_bytes())
    want = dict(CFBundleIdentifier=t.identifier, CFBundleExecutable=t.name, CFBundlePackageType='APPL')
    why += [f'{k} is {p.get(k)!r}, not {v!r}' for k, v in want.items() if p.get(k) != v]
    if not p.get('NSAppTransportSecurity', {}).get('NSAllowsLocalNetworking'): why.append('no NSAllowsLocalNetworking')
    if (t.doc_types or self.hook('plist')) and not p.get('CFBundleDocumentTypes'): why.append('no document types')
    if not os.access(c/'MacOS'/t.name, os.X_OK): why.append('no launcher')
    uv = c/'Helpers'/'uv'
    if not os.access(uv, os.X_OK): why.append('no Helpers/uv')
    elif (v := uv_version(uv)) != self.pinned_version(): why.append(f'Helpers/uv is {v}, not {self.pinned_version()}')
    try: m = self.check_lock_dir(c/'Resources'/'Lock', c/'Resources'/'lock-keys.pub')
    except (SystemExit, OSError, ValueError) as e: why.append(f'bootstrap lock: {e}'); m = {}
    for f in ('childenv.json', 'kavacha.json'):
        if not (c/'Resources'/f).is_file(): why.append(f'no {f}')
    if (c/'Resources'/'kavacha.json').is_file() and (k := json.loads((c/'Resources'/'kavacha.json').read_text())) != json.loads(json.dumps(t.config())): why.append('kavacha.json is not this spec')
    dev = not self.keys_pub.is_file() or (c/'Resources'/'lock-keys.pub').read_text() != self.keys_pub.read_text()
    kb = sum(f.stat().st_size for f in app.rglob('*') if f.is_file() and not f.is_symlink()) // 1024
    zkb = Path(zip_).stat().st_size // 1024 if zip_ else 0
    if kb > t.max_mb * 1024: why.append(f'{kb/1024:.1f} MB on disk is over {t.max_mb} MB')
    if zkb > t.max_zip_mb * 1024: why.append(f'{zkb/1024:.1f} MB zipped is over {t.max_zip_mb} MB')
    if why: raise SystemExit('the app is refused:\n  ' + '\n  '.join(why))
    print(f"bundle ok: {p.get('CFBundleShortVersionString')} ({p.get('CFBundleVersion')}), uv {self.pinned_version()}, lock {m.get('n')} ({t.package} {m.get(t.package)}), {kb/1024:.1f} MB"
          + (f', {zkb/1024:.1f} MB zipped' if zip_ else '') + (', signed with a throwaway key: not for release' if dev else ''))
    return dict(kb=kb, zip_kb=zkb, dev=dev, lock=m.get('n'))
Lock.verify_app = _verify_app

### The command line

`kavacha lock <command>` runs in the app's folder (or `--root`). Each command is one method above.

In [ ]:
#| export
def main(argv=None):
    "kavacha lock make|hash|check|test|diff|publish|promote|verify|key|dev-lock|fetch|wheel|fixtures|verify-app"
    a = argparse.ArgumentParser(prog='kavacha lock', description='The signed environment lock of a thin app.')
    a.add_argument('--root', default='.')
    s = a.add_subparsers(dest='cmd', required=True)
    p = s.add_parser('make'); p.add_argument('version'); p.add_argument('--dir'); p.add_argument('--wheel'); p.add_argument('--pin', action='append', default=[])
    for n in ('hash', 'check'): s.add_parser(n).add_argument('--dir')
    p = s.add_parser('test'); p.add_argument('--dir'); p.add_argument('--fresh', action='store_true'); p.add_argument('--extras', action='store_true'); p.add_argument('--records'); p.add_argument('--keep', action='store_true')
    p = s.add_parser('diff'); p.add_argument('--dir'); p.add_argument('--against')
    s.add_parser('publish').add_argument('--allow-downgrade', action='store_true')
    s.add_parser('promote').add_argument('n', type=int)
    s.add_parser('verify').add_argument('--channel', default='stable', choices=['stable', 'test'])
    s.add_parser('key').add_argument('--keys-out')
    p = s.add_parser('dev-lock'); p.add_argument('out'); p.add_argument('--from', dest='src'); p.add_argument('--n', type=int, default=0)
    p = s.add_parser('fetch'); p.add_argument('out'); p.add_argument('--channel', default='stable', choices=['stable', 'test'])
    p = s.add_parser('wheel'); p.add_argument('version'); p.add_argument('--upload', action='store_true')
    p = s.add_parser('fixtures'); p.add_argument('out'); p.add_argument('--ns', default='app-lock')
    p = s.add_parser('verify-app'); p.add_argument('app'); p.add_argument('--zip')
    o = a.parse_args(argv)
    if o.cmd == 'fixtures': return fixtures(o.out, o.ns)
    lk = Lock(Thin.load(o.root))
    ch = lambda: 'channel-test' if getattr(o, 'channel', '') == 'test' else 'channel'
    match o.cmd:
        case 'make': lk.make(o.version, o.dir, o.wheel, pins=o.pin)
        case 'hash': print(lock_hash(o.dir or lk.lock, lk.pinned_version()))
        case 'check':
            if why := lk.check(o.dir or lk.lock): raise SystemExit('\n'.join(why))
            print('lock ok')
        case 'test':
            if not lk.lock_test(o.dir, o.fresh, o.extras, o.records, o.keep)['passed']: raise SystemExit(1)
        case 'diff': lk.diff(o.dir, old=packages(o.against) if o.against else None)
        case 'publish': lk.publish(allow_downgrade=o.allow_downgrade)
        case 'promote': lk.promote(o.n)
        case 'verify': lk.verify(ch())
        case 'key': lk.lock_key(keys_out=o.keys_out or lk.keys_pub)
        case 'dev-lock': lk.dev_lock(o.out, o.src, o.n)
        case 'fetch': lk.fetch_channel(o.out, ch())
        case 'wheel': lk.wheel(o.version, o.upload)
        case 'verify-app': lk.verify_app(o.app, o.zip)